In [ ]:
#!git clone https://github.com/whyhardt/SPICE.git

In [ ]:
# !pip install -e SPICE

In [1]:
import sys
import torch

from spice import SpiceEstimator, SpiceConfig, SpiceDataset, csv_to_dataset, BaseRNN, split_data_along_sessiondim

sys.path.append('../../..')
from weinhardt2026.utils.benchmarking_gru import Model as GRU, training

Let's load the data first with the `convert_dataset` method. This method returns a `SpiceDataset` object which we can use right away 

In [9]:
# Load your data
# Load your data
dataset = csv_to_dataset(
    file = '../data/hwang2025/hwang2025_processed.csv',
    df_participant_id='interaction_id',
    df_choice='SigAct_ID1',
    df_feedback=None,
    additional_inputs=['SigAct_ID2', 'ID1', 'ID2'],
    )

# in order to set up the participant embedding we have to compute the number of unique participants in our data 
# to get the number of participants n_participants we do:
n_actions = dataset.ys.shape[-1]

# replace participant id and experiment id columns in dataset with ID1 and ID2 columns from additional inputs
# participant id -> ID1
# experiment id -> ID2
dataset.xs[..., -1] = dataset.xs[..., n_actions+1].nan_to_num(0)
dataset.xs[..., -2] = dataset.xs[..., n_actions+2].nan_to_num(0)

n_participants = dataset.xs[:, 0, -1].nan_to_num(0).max().int().item()+1
    
# structure of dataset:
# dataset has two main attributes: xs -> inputs; ys -> targets (next action)
# shape: (n_participants*n_blocks*n_experiments, n_timesteps, features)
# features are (n_actions * action, n_actions * reward, n_additional_inputs * additional_input, block_number, experiment_id, participant_id)

print(f"Shape of dataset: {dataset.xs.shape}")
print(f"Number of participants: {n_participants}")
print(f"Number of actions in dataset: {n_actions}")
print(f"Number of additional inputs: {dataset.xs.shape[-1]-n_actions-5}")

Shape of dataset: torch.Size([311, 144, 1, 12])
Number of participants: 42
Number of actions in dataset: 4
Number of additional inputs: 3


In [10]:
# 12 blocks per participants
# 75 % -> Training
# 25 % -> Testing
# Excluding in testing w.r.t. blocks -> no blocks from beginning (e.g. task learning); no blocks from the end (e.g. fatigue)
#   -> test_blocks = 3, 6, 9
#   -> training_blocks = 0, 1, 2, 4, 5, 7, 10, 11

test_blocks = (1,)
dataset_train, dataset_test = split_data_along_sessiondim(dataset=dataset, list_test_sessions=test_blocks)

print(f"Ratio of test/train: {len(dataset_test) / len(dataset_train):.2f}")

Ratio of test/train: 0.33


### Dataset description

In [27]:
dataset.xs.shape # shape -> (n_participants: 41, timesteps: 496, features: 16)

# normal RL exp:    [A] [B] [C] [D] [E]
# choice:           [x] [ ] [ ] [ ] [ ]
# reward:           [1] [ ] [ ] [ ] [ ]    (partial feedback)
# reward:           [1] [0] [1] [1] [0]    (full feedback)

# features: (action0, action1, action2, action3, action4, reward0, reward1, reward2, reward3, reward4, 'ID2', 'SigAct_ID2', 'Grooming_ID1', block number, experiment id, ID1)
# in your case: (x, x, x, x, x, -, -, -, -, -, x, x, x, -, -, -, -, x)    -> x: keep; -: ignore

torch.Size([311, 144, 1, 12])

Now we are going to define the configuration for SPICE with a `SpiceConfig` object.

The `SpiceConfig` takes as arguments 
1. `library_setup (dict)`: Defining the variable names of each module.
2. `memory_state (dict)`: Defining the memory state variables and their initial values.
3. `states_in_logit (list)`: Defining which of the memory state variables are used later for the logit computation. This is necessary for some background processes.  

In [28]:
spice_config = SpiceConfig(
    library_setup={
        'module_action': ['action_id1', 'grooming_id1', 'gesture_id1', 'action_id2', 'grooming_id2', 'gesture_id2'],
        'module_grooming': ['action_id1', 'grooming_id1', 'gesture_id1', 'action_id2', 'grooming_id2', 'gesture_id2'],
        'module_gesture': ['action_id1', 'grooming_id1', 'gesture_id1', 'action_id2', 'grooming_id2', 'gesture_id2'],
        },
    
    memory_state={
        'values': 0,
        },
)

And now we are going to define the SPICE model which is a child of the `BaseRNN` and `torch.nn.Module` class and takes as required arguments:
1. `spice_config (SpiceConfig)`: previously defined SpiceConfig object
2. `n_actions (int)`: number of possible actions in your dataset (including non-displayed ones if applicable).
3. `n_participants (int)`: number of participants in your dataset.

As usual for a `torch.nn.Module` we have to define at least the `__init__` method and the `forward` method.
The `forward` method gets called when computing a forward pass through the model and takes as inputs `(inputs (SpiceDataset.xs), prev_state (dict, default: None), batch_first (bool, default: False))` and returns `(logits (torch.Tensor, shape: (n_participants*n_blocks*n_experiments, timesteps, n_actions)), updated_state (dict))`. Two necessary method calls inside the forward pass are:
1. `self.init_forward_pass(inputs, prev_state, batch_first) -> SpiceSignals`: returns a `SpiceSignals` object which carries all relevant information already processed.
2. `self.post_forward_pass(SpiceSignals, batch_first) -> SpiceSignals`: does some re-arranging of the logits to adhere to `batch_first`.

In [29]:
class SpiceModel(BaseRNN):
    
    def __init__(self, **kwargs):
        super().__init__(embedding_size=8, **kwargs)
        
        dropout = 0.1
            
        # participant embedding
        self.participant_embedding = self.setup_embedding(num_embeddings=self.n_participants, embedding_size=self.embedding_size, dropout=dropout)
        
        # rnn modules
        self.setup_module(key_module='module_action', input_size=6+self.embedding_size*2, dropout=dropout)
        self.setup_module(key_module='module_grooming', input_size=6+self.embedding_size*2, dropout=dropout)
        self.setup_module(key_module='module_gesture', input_size=6+self.embedding_size*2, dropout=dropout)
    
    def forward(self, inputs, prev_state=None, batch_first=False):
        
        spice_signals = self.init_forward_pass(inputs, prev_state, batch_first)
        
        # time-invariant participant features
        participant_embeddings_id1 = self.participant_embedding(spice_signals.participant_ids)
        participant_embeddings_id2 = self.participant_embedding(spice_signals.experiment_ids)
        
        # feature extraction
        # id 1
        action_id1 = spice_signals.actions[..., 0].unsqueeze(-1).expand_as(spice_signals.actions)
        grooming_id1 = spice_signals.actions[..., 1].unsqueeze(-1).expand_as(spice_signals.actions)
        gesture_id1 = spice_signals.actions[..., 2].unsqueeze(-1).expand_as(spice_signals.actions)
        # id 2
        actions_id2 = torch.eye(self.n_actions, device=self.device)[spice_signals.additional_inputs[..., 0].int()]  # one hot encoding of actions id2
        action_id2 = actions_id2[..., 0].unsqueeze(-1).expand_as(spice_signals.actions)
        grooming_id2 = actions_id2[..., 1].unsqueeze(-1).expand_as(spice_signals.actions)
        gesture_id2 = actions_id2[..., 2].unsqueeze(-1).expand_as(spice_signals.actions)
        
        # action masks
        mask_action = torch.tensor((1, 0, 0, 0), device=self.device).reshape(1, 1, 1, -1).expand_as(spice_signals.actions)
        mask_grooming = torch.tensor((0, 1, 0, 0), device=self.device).reshape(1, 1, 1, -1).expand_as(spice_signals.actions)
        mask_gesture = torch.tensor((0, 0, 1, 0), device=self.device).reshape(1, 1, 1, -1).expand_as(spice_signals.actions)
        
        for timestep in spice_signals.trials:
            
            # update chosen value
            self.call_module(
                key_module='module_action',
                key_state='values',
                action_mask=mask_action, 
                inputs=(
                    action_id1[timestep],
                    grooming_id1[timestep], 
                    gesture_id1[timestep],
                    action_id2[timestep],
                    grooming_id2[timestep], 
                    gesture_id2[timestep],
                    ),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embeddings_id1,
                experiment_index=spice_signals.experiment_ids,
                experiment_embedding=participant_embeddings_id2,
            )
            
            self.call_module(
                key_module='module_grooming',
                key_state='values',
                action_mask=mask_grooming, 
                inputs=(
                    action_id1[timestep],
                    grooming_id1[timestep], 
                    gesture_id1[timestep],
                    action_id2[timestep],
                    grooming_id2[timestep], 
                    gesture_id2[timestep],
                    ),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embeddings_id1,
                experiment_index=spice_signals.experiment_ids,
                experiment_embedding=participant_embeddings_id2,
            )
            
            self.call_module(
                key_module='module_gesture',
                key_state='values',
                action_mask=mask_gesture, 
                inputs=(
                    action_id1[timestep],
                    grooming_id1[timestep], 
                    gesture_id1[timestep],
                    action_id2[timestep],
                    grooming_id2[timestep], 
                    gesture_id2[timestep],
                    ),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embeddings_id1,
                experiment_index=spice_signals.experiment_ids,
                experiment_embedding=participant_embeddings_id2,
            )
            
            spice_signals.logits[timestep] = self.state['values']
                
        spice_signals = self.post_forward_pass(spice_signals, batch_first)
        
        return spice_signals.logits, self.get_state()

In [30]:
def cross_entropy_loss_mask_waiting(prediction, target):
    
    n_actions = target.shape[-1]
    
    prediction = prediction.reshape(-1, n_actions)
    target = torch.argmax(target.reshape(-1, n_actions), dim=1)
    
    non_waiting_mask = target != 3  # filters where ID1 is just waiting -> that way we are dropping waiting as an action category
    
    return torch.nn.functional.cross_entropy(prediction[non_waiting_mask], target[non_waiting_mask])

In [ ]:
path_spice = '../params/hwang2025/spice_hwang2025.pkl'
estimator = SpiceEstimator(
    spice_class=SpiceModel,
    spice_config=spice_config,
    loss_fn=cross_entropy_loss_mask_waiting,
    
    n_actions=n_actions,
    n_participants=n_participants,
    n_experiments=n_participants,
    n_reward_features=0,
    
    epochs=1000,
    warmup_steps=200,
    ensemble_size=10,
    
    sindy_weight=0.,  # for RNN-only training: sindy_weight = 0; for RNN+SINDy training: sindy_weight=0.1
    sindy_alpha=0.0001,
    
    sindy_library_polynomial_degree=2,
    sindy_pruning_frequency=100,
    sindy_threshold_pruning=0.01,
    sindy_ensemble_pruning=0.05,
    sindy_refit=True,
    
    verbose=True,
    save_path_spice=path_spice,
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
)

In [32]:
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

 22%|█████████████▋                                                 | 217/1000 [01:45<06:21,  2.05it/s, L(Train)=0.8127192, L(Val,RNN)=1.0548309, Conv=1.77e-03]

Training interrupted. Continuing with further operations...

Training results:
	L(Train, RNN): 0.0000000
	L(Val, RNN):   1.0548309

RNN training finished.
Training took 106.13 seconds.
Saving SPICE model to ../params/hwang2025/spice_hwang2025.pkl...


In [9]:
estimator.load_spice(path_spice)

In [11]:
print(estimator.get_sindy_coefficients()['module_action'].shape)  # shape of coefficients: (ID1, ID2, Ensemble, Coefficients)

torch.Size([10, 42, 42, 8])


In [12]:
estimator.get_candidate_terms('module_action')


['1',
 'module_action',
 'action_id1',
 'grooming_id1',
 'gesture_id1',
 'action_id2',
 'grooming_id2',
 'gesture_id2']

In [13]:
len(estimator.get_candidate_terms('module_action'))

8

In [15]:
# Be careful! not all Ape-pairings were actually recorded!

print("SPICE for Ape 0 -> Ape 2:")
estimator.print_spice_model(participant_id=0, experiment_id=2)

print("\nSPICE for Ape 0 -> Ape 5:")
estimator.print_spice_model(participant_id=0, experiment_id=5)

print("\nSPICE for Ape 37 -> Ape 10:")
estimator.print_spice_model(participant_id=37, experiment_id=10)

SPICE for Ape 0 -> Ape 2:
module_action[t+1]   = -0.0 1 + 1.0 module_action[t] + 0.0 action_id1 + -0.0 grooming_id1 + 0.0 gesture_id1 + 0.0 action_id2 + -0.0 grooming_id2 + 0.0 gesture_id2 
module_grooming[t+1] = -0.0 1 + 1.0 module_grooming[t] + -0.0 action_id1 + 0.0 grooming_id1 + -0.0 gesture_id1 + -0.0 action_id2 + -0.001 grooming_id2 + -0.0 gesture_id2 
module_gesture[t+1]  = 0.0 1 + 1.0 module_gesture[t] + 0.0 action_id1 + -0.0 grooming_id1 + 0.0 gesture_id1 + -0.001 action_id2 + 0.0 grooming_id2 + -0.0 gesture_id2 

SPICE for Ape 0 -> Ape 5:
module_action[t+1]   = 0.0 1 + 1.0 module_action[t] + -0.0 action_id1 + 0.0 grooming_id1 + 0.0 gesture_id1 + -0.0 action_id2 + -0.001 grooming_id2 + 0.001 gesture_id2 
module_grooming[t+1] = -0.0 1 + 1.001 module_grooming[t] + -0.0 action_id1 + 0.0 grooming_id1 + -0.0 gesture_id1 + -0.0 action_id2 + 0.0 grooming_id2 + -0.0 gesture_id2 
module_gesture[t+1]  = -0.0 1 + 1.0 module_gesture[t] + 0.0 action_id1 + -0.0 grooming_id1 + 0.0 gesture_id

## GRU for benchmarking

### Classic GRU

In [ ]:
gru = GRU(n_actions=n_actions, additional_inputs=3, n_reward_features=0)
path_gru = '../params/hwang2025/gru_hwang2025.pkl'

In [16]:
epochs = 1000
optimizer = torch.optim.Adam(gru.parameters(), lr=0.01)

gru = training(
    model=gru,
    optimizer=optimizer,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
    criterion=cross_entropy_loss_mask_waiting,
    ).to(torch.device('cpu'))

torch.save(gru.state_dict(), path_gru)
print("Trained GRU parameters saved to " + path_gru)

Epoch 1/1000: L(Train): 1.7420872449874878; L(Test): 1.3949460983276367
Epoch 2/1000: L(Train): 1.4553067684173584; L(Test): 1.2746696472167969
Epoch 3/1000: L(Train): 1.2949035167694092; L(Test): 1.2359590530395508
Epoch 4/1000: L(Train): 1.2316796779632568; L(Test): 1.205856442451477
Epoch 5/1000: L(Train): 1.1931668519973755; L(Test): 1.1799519062042236
Epoch 6/1000: L(Train): 1.1619106531143188; L(Test): 1.1630244255065918
Epoch 7/1000: L(Train): 1.144761323928833; L(Test): 1.1521291732788086
Epoch 8/1000: L(Train): 1.1293551921844482; L(Test): 1.1437956094741821
Epoch 9/1000: L(Train): 1.1199179887771606; L(Test): 1.1356959342956543
Epoch 10/1000: L(Train): 1.1158760786056519; L(Test): 1.1268646717071533
Epoch 11/1000: L(Train): 1.1035454273223877; L(Test): 1.1174426078796387
Epoch 12/1000: L(Train): 1.0992945432662964; L(Test): 1.1079424619674683
Epoch 13/1000: L(Train): 1.0950603485107422; L(Test): 1.099541425704956
Epoch 14/1000: L(Train): 1.0883688926696777; L(Test): 1.0933831

In [32]:
gru.load_state_dict(torch.load(path_gru))

<All keys matched successfully>

### GRU with participant embedding

In [23]:
class GRUEmbed(torch.nn.Module):
    
    def __init__(self, n_actions, n_participants, additional_inputs: int = 0, n_reward_features: int = None, hidden_size: int = 32, dropout: float = 0., **kwargs):
        super().__init__()
        
        self.gru_features = hidden_size
        self.n_actions = n_actions
        self.additional_inputs = additional_inputs
        self.n_reward_features = n_reward_features if n_reward_features is not None else n_actions
        self.embed_size = 4
        
        self.embedding = torch.nn.Embedding(n_participants, self.embed_size)
        
        self.linear_in = torch.nn.Linear(in_features=n_actions+n_reward_features+additional_inputs+2*self.embed_size, out_features=hidden_size)
        self.dropout = torch.nn.Dropout(dropout)
        self.gru = torch.nn.GRU(input_size=hidden_size, hidden_size=hidden_size, batch_first=True)
        self.linear_out = torch.nn.Linear(in_features=hidden_size, out_features=n_actions)
        
    def forward(self, inputs, state=None):
        
        id1 = inputs[..., -1].nan_to_num(0).long()
        id2 = inputs[..., -2].nan_to_num(0).long()
        
        embed1 = self.embedding(id1)
        embed2 = self.embedding(id2)
        
        actions = inputs[..., :self.n_actions]
        rewards = inputs[..., self.n_actions:self.n_actions+self.n_reward_features].nan_to_num(0)#.sum(dim=-1, keepdims=True)
        additional_inputs = inputs[..., self.n_actions+self.n_reward_features:self.n_actions+self.n_reward_features+self.additional_inputs]
        inputs = torch.concat((actions, rewards, additional_inputs, embed1, embed2), dim=-1)
        
        if state is not None and len(inputs.shape) == 3:
            state = state.reshape(1, 1, self.gru_features)
        
        y = self.linear_in(inputs[:, :, 0].nan_to_num(0))
        y = self.dropout(y)
        y, state = self.gru(y, state)
        y = self.dropout(y)
        y = self.linear_out(y)
        return y.unsqueeze(2), state


In [24]:
path_gru_embed = '../../weinhardt2026/params/hwang2025/gruembed_hwang2025.pkl'
gru_embed = GRUEmbed(n_actions=n_actions, additional_inputs=4, n_participants=n_participants, n_reward_features=0, dropout=0.2)

In [ ]:
epochs = 1000
optimizer = torch.optim.Adam(gru_embed.parameters(), lr=0.01)

gru = training(
    model=gru_embed,
    optimizer=optimizer,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
    criterion=cross_entropy_loss_mask_waiting,
    ).to(torch.device('cpu'))

torch.save(gru_embed.state_dict(), path_gru_embed)
print("Trained GRU+Embedding parameters saved to " + path_gru_embed)

Epoch 1/1000: L(Train): 0.7705872654914856; L(Test): 0.7766693234443665
Epoch 2/1000: L(Train): 0.8648998737335205; L(Test): 0.706669807434082
Epoch 3/1000: L(Train): 0.7778975367546082; L(Test): 0.7463822960853577
Epoch 4/1000: L(Train): 0.8154776096343994; L(Test): 0.7406186461448669
Epoch 5/1000: L(Train): 0.8014295101165771; L(Test): 0.724433422088623
Epoch 6/1000: L(Train): 0.7817184329032898; L(Test): 0.7131935954093933
Epoch 7/1000: L(Train): 0.7819126844406128; L(Test): 0.70636385679245
Epoch 8/1000: L(Train): 0.7723742723464966; L(Test): 0.7079607248306274
Epoch 9/1000: L(Train): 0.7801750302314758; L(Test): 0.7111022472381592
Epoch 10/1000: L(Train): 0.7856540083885193; L(Test): 0.7096582651138306
Epoch 11/1000: L(Train): 0.7763304710388184; L(Test): 0.7063232660293579
Epoch 12/1000: L(Train): 0.7639082074165344; L(Test): 0.7023528814315796
Epoch 13/1000: L(Train): 0.7743151187896729; L(Test): 0.6980308890342712
Epoch 14/1000: L(Train): 0.7559747099876404; L(Test): 0.69540852

In [28]:
gru_embed.load_state_dict(torch.load(path_gru_embed, map_location='cpu'))

<All keys matched successfully>

# ANALYSIS

In [34]:
from weinhardt2026.analysis.analysis_model_evaluation import analysis_model_evaluation
from weinhardt2026.analysis.analysis_coefficients_individuals import analysis_coefficients_individuals
from weinhardt2026.analysis.analysis_coefficients_distributions import analysis_coefficients_distributions

In [35]:
analysis_model_evaluation(
    dataset=dataset_test,
    spice_model=estimator,
    gru_model=gru_embed,
)

Computing choice probabilities with GRU model...
Computing choice probabilities with SPICE model...


ValueError: only one element tensors can be converted to Python scalars